In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install easyocr opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 20.3 MB/s eta 0:00:00


In [3]:
# Cell 3 — Upload your aggregated_sales.csv from Step 1
from google.colab import files
uploaded = files.upload()

Saving aggregated_sales.csv to aggregated_sales.csv


In [ ]:
# Cell 4 — Run OCR
import easyocr
import re
import os
import cv2
import pandas as pd

# Load known codes
df_agg = pd.read_csv("aggregated_sales.csv")
known_codes = df_agg["code"].astype(str).tolist()
print(f"Looking for {len(known_codes)} codes")

# Your images folder path in Drive
image_folder = '/content/drive/MyDrive/maalde_images/'  # ← change if folder name is different

reader = easyocr.Reader(['en'], gpu=True)

records = []

all_images = [f for f in os.listdir(image_folder)
              if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

print(f"Total images found: {len(all_images)}")

for i, fname in enumerate(all_images):
    img_path = os.path.join(image_folder, fname)

    try:
        # Crop bottom 15% only — faster OCR
        img = cv2.imread(img_path)
        h, w = img.shape[:2]
        cropped = img[int(h * 0.85):h, :]

        results = reader.readtext(cropped, detail=0)
        text = " ".join(results)
        numbers = re.findall(r'\d{5,}', text)

        matched_code = None
        for num in numbers:
            if num in known_codes:
                matched_code = num
                break

        records.append({
            "filename": fname,
            "ocr_text": text,
            "matched_code": matched_code
        })

        status = f"✅ {matched_code}" if matched_code else "❌ no match"
        print(f"[{i+1}/{len(all_images)}] {fname} → {status}")

    except Exception as e:
        print(f"⚠️ Error on {fname}: {e}")
        records.append({
            "filename": fname,
            "ocr_text": "",
            "matched_code": None
        })

    # Save after every 10 images — safety from disconnect
    if (i + 1) % 10 == 0:
        pd.DataFrame(records).to_csv(
            '/content/drive/MyDrive/ocr_mapping.csv', index=False)
        print(f"💾 Progress saved at {i+1} images")

# Final save
df_result = pd.DataFrame(records)
df_result.to_csv('/content/drive/MyDrive/ocr_mapping.csv', index=False)

matched = df_result["matched_code"].notna().sum()
print(f"\n✅ DONE!")
print(f"Matched: {matched}/{len(df_result)} images")
print(f"\nUnmatched images ({len(df_result)-matched}):")
print(df_result[df_result["matched_code"].isna()]["filename"].tolist())

Looking for 146 codes
Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% CompleteTotal images found: 180


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[1/180] WhatsApp Image 2026-04-29 at 1.24.06 PM.jpeg → ❌ no match
[2/180] WhatsApp Image 2026-04-29 at 1.24.08 PM (1).jpeg → ✅ 10029365
[3/180] WhatsApp Image 2026-04-29 at 1.24.07 PM.jpeg → ✅ 10029353
[4/180] WhatsApp Image 2026-04-29 at 1.24.06 PM (1).jpeg → ❌ no match
[5/180] WhatsApp Image 2026-04-29 at 1.24.08 PM.jpeg → ✅ 10028810
[6/180] WhatsApp Image 2026-04-29 at 1.24.06 PM (2).jpeg → ❌ no match
[7/180] WhatsApp Image 2026-04-29 at 1.24.07 PM (1).jpeg → ❌ no match
[8/180] WhatsApp Image 2026-04-29 at 1.24.05 PM (1).jpeg → ✅ 10029463
[9/180] WhatsApp Image 2026-04-29 at 1.24.05 PM.jpeg → ✅ 10027912
[10/180] WhatsApp Image 2026-04-29 at 1.24.39 PM (1).jpeg → ❌ no match
💾 Progress saved at 10 images
[11/180] WhatsApp Image 2026-04-29 at 1.24.29 PM.jpeg → ❌ no match
[12/180] WhatsApp Image 2026-04-29 at 1.24.17 PM (1).jpeg → ✅ 10029369
[13/180] WhatsApp Image 2026-04-29 at 1.24.09 PM (1).jpeg → ❌ no match
[14/180] WhatsApp Image 2026-04-29 at 1.24.40 PM.jpeg → ❌ no match
[15/180] 

In [5]:
from google.colab import files
files.download('/content/drive/MyDrive/ocr_mapping.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
import pandas as pd

# Load both files
ocr = pd.read_csv('/content/drive/MyDrive/ocr_mapping.csv', low_memory=False)
sales = pd.read_csv('aggregated_sales.csv')

# Check what we have
print("OCR columns:", ocr.columns.tolist())
print("OCR shape:", ocr.shape)
print("\nSample matched_code values:")
print(ocr["matched_code"].head(20))
print("\nUnique matched_code types:", ocr["matched_code"].dtype)

OCR columns: ['filename', 'ocr_text', 'matched_code']
OCR shape: (180, 3)

Sample matched_code values:
0       500001.0
1     10019275.0
2     10021130.0
3     10021131.0
4     10022091.0
5     10022578.0
6     10025103.0
7     10025105.0
8     10025542.0
9     10025820.0
10    10025821.0
11    10026674.0
12    10027276.0
13    10027278.0
14    10027279.0
15    10027280.0
16    10027298.0
17    10027299.0
18    10027542.0
19    10027574.0
Name: matched_code, dtype: float64

Unique matched_code types: float64


In [11]:
import pandas as pd
import numpy as np

# Load files
ocr = pd.read_csv('/content/drive/MyDrive/ocr_mapping.csv', low_memory=False)
sales = pd.read_csv('aggregated_sales.csv')

# Fix float codes 500001.0 → "500001"
ocr["matched_code"] = pd.to_numeric(ocr["matched_code"], errors='coerce')
ocr["matched_code"] = ocr["matched_code"].astype('Int64').astype(str)
ocr["matched_code"] = ocr["matched_code"].replace("<NA>", np.nan)

# Drop unmatched
ocr_matched = ocr.dropna(subset=["matched_code"]).copy()
print(f"Matched images: {len(ocr_matched)}")

# Clean sales codes
sales["code"] = sales["code"].astype(str).str.strip()
ocr_matched["matched_code"] = ocr_matched["matched_code"].astype(str).str.strip()

# Merge
final_df = ocr_matched.merge(
    sales,
    left_on="matched_code",
    right_on="code",
    how="inner"
)

print(f"✅ Final dataset: {len(final_df)} images ready for training")
print(final_df[["filename", "matched_code", "total_qty", "avg_rate"]].head(10))

# Save
final_df.to_csv('/content/drive/MyDrive/final_dataset.csv', index=False)

Matched images: 129
✅ Final dataset: 129 images ready for training
                                           filename matched_code  total_qty  \
0  WhatsApp Image 2026-04-29 at 1.25.57 PM (2).jpeg       500001       1032   
1      WhatsApp Image 2026-04-29 at 1.26.59 PM.jpeg     10019275         56   
2  WhatsApp Image 2026-04-29 at 1.27.09 PM (1).jpeg     10021130         32   
3      WhatsApp Image 2026-04-29 at 1.27.10 PM.jpeg     10021131         28   
4      WhatsApp Image 2026-04-29 at 1.26.57 PM.jpeg     10022091         84   
5      WhatsApp Image 2026-04-29 at 1.26.56 PM.jpeg     10022578         60   
6      WhatsApp Image 2026-04-29 at 1.25.57 PM.jpeg     10025103         60   
7  WhatsApp Image 2026-04-29 at 1.27.01 PM (1).jpeg     10025105         47   
8  WhatsApp Image 2026-04-29 at 1.22.59 PM (1).jpeg     10025542         71   
9  WhatsApp Image 2026-04-29 at 1.26.58 PM (1).jpeg     10025820         28   

      avg_rate  
0  1295.017544  
1  1650.000000  
2   875.0000

In [12]:
import numpy as np
import os
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing import image as keras_image

# Load MobileNetV2
mobilenet = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')
print("✅ MobileNetV2 loaded")

image_folder = '/content/drive/MyDrive/maalde_images/'

features = []
targets = []
valid_files = []

for i, row in final_df.iterrows():
    img_path = os.path.join(image_folder, row["filename"])

    try:
        img = keras_image.load_img(img_path, target_size=(224, 224))
        x = keras_image.img_to_array(img)
        x = np.expand_dims(x, axis=0)
        x = preprocess_input(x)

        feat = mobilenet.predict(x, verbose=0)
        features.append(feat.flatten())
        targets.append(row["total_qty"])
        valid_files.append(row["filename"])

        print(f"✅ [{len(features)}/129] {row['filename']} → qty: {row['total_qty']}")

    except Exception as e:
        print(f"⚠️ Skipped {row['filename']}: {e}")

print(f"\n✅ Features extracted: {len(features)} images")

/tmp/ipykernel_1112/4278165071.py:8: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  mobilenet = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
✅ MobileNetV2 loaded
✅ [1/129] WhatsApp Image 2026-04-29 at 1.25.57 PM (2).jpeg → qty: 1032
✅ [2/129] WhatsApp Image 2026-04-29 at 1.26.59 PM.jpeg → qty: 56
✅ [3/129] WhatsApp Image 2026-04-29 at 1.27.09 PM (1).jpeg → qty: 32
✅ [4/129] WhatsApp Image 2026-04-29 at 1.27.10 PM.jpeg → qty: 28
✅ [5/129] WhatsApp Image 2026-04-29 at 1.26.57 PM.jpeg → qty: 84
✅ [6/129] WhatsApp Image 2026-04-29 at 1.26.56 PM.jpeg → qty: 60
✅ [7/129] WhatsApp Image 2026-04-29 at 1.25.57 PM.jpeg → qty: 60
✅ [8/129] WhatsApp Image 2026-04-29 at 1.27.01 PM (1).jpeg → qty: 47
✅ [9/129] WhatsApp Image 2026-04-29 at 1.22.59 PM (1).jpeg → qty: 71
✅ [10/129] WhatsApp Image 2026-04-29 at 1.26.58 PM (1).jpeg → qty: 28
✅ [11/129] WhatsApp Image 2026-04-29 at 1.26.58 PM.jpeg → qty: 52
✅ [12/129] WhatsApp Image 2026-04-29 at 1.25.48 PM.jpeg → qty: 36
✅ [13/129] WhatsApp Image 2026-04-29 at 1.22.59 PM.jpeg → qty: 12
✅ [14/129] WhatsApp Image 2026-04-29 at 1.23.09 PM (2).jpeg

In [13]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import numpy as np

X = np.array(features)
y = np.array(targets)

print(f"Training on {len(X)} images")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# RandomForest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_preds)

# GradientBoosting
gb = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)
gb_preds = gb.predict(X_test)
gb_mae = mean_absolute_error(y_test, gb_preds)

print(f"\nRandomForest MAE:     {rf_mae:.2f}")
print(f"GradientBoosting MAE: {gb_mae:.2f}")

# Pick best
best_model = rf if rf_mae < gb_mae else gb
best_name = "RandomForest" if rf_mae < gb_mae else "GradientBoosting"
print(f"\n✅ Best model: {best_name}")

Training on 129 images

RandomForest MAE:     23.08
GradientBoosting MAE: 13.83

✅ Best model: GradientBoosting


In [ ]:
import joblib
import shutil

# Save model
joblib.dump(best_model, 'demand_model.pkl')
np.save('features.npy', X)
np.save('targets.npy', y)

# Copy to Drive
shutil.copy('demand_model.pkl', '/content/drive/MyDrive/demand_model.pkl')
shutil.copy('features.npy', '/content/drive/MyDrive/features.npy')
shutil.copy('targets.npy', '/content/drive/MyDrive/targets.npy')

print("✅ All saved to Drive!")

✅ All saved to Drive!


In [16]:
from google.colab import files
files.download('demand_model.pkl')
files.download('features.npy')
files.download('targets.npy')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>